# Physics-LLM Adapter Training on Google Colab (v2 - Expanded Data)

This notebook trains the Physics-LLM Adapter model using Colab's GPU.

**Training on expanded CLEVRER dataset: 152,736 samples in HDF5 format**

**Features:**
- Checkpoint saving every 3 epochs
- Checkpoint saving at end of each phase
- All checkpoints save to Google Drive
- Resume from any checkpoint
- Multi-head descriptive classifier for CLEVRER
- HDF5 dataset for efficient loading of 152K samples

**Training Phases:**
1. **Phase 1**: Adapter MLP + Numerical + Descriptive heads (LLM frozen)
2. **Phase 2**: LoRA on LLM attention + contrastive loss (recovers the Jan 25 `adapter_v2_distilgpt2_contrastive.pt` recipe; required for the LLM to actually use the physics prefix tokens during MCQ scoring)
3. **Phase 3**: Full-LLM fine-tuning (optional; usually unnecessary once Phase 2 LoRA is trained)

**Recommended multi-session flow:**
- Session 1: Run Cells 1-10, 13 (Phase 1 only), download `adapter_v2_expanded_final.pt`
- Session 2: Set `RESUME_CHECKPOINT` + `START_PHASE=2` in Cell 9, run Cells 1-9, 11, 13 (Phase 2 LoRA), download

## Prerequisites
1. Upload `physics_llm_colab.zip` to Google Drive (`/MyDrive/physics_llm/`)
2. Upload `clevrer_training_expanded.h5` (1.6GB) to `/MyDrive/physics_llm/data/`
3. Upload `physics_former_best.pt` to `/MyDrive/physics_llm/checkpoints/`
4. Enable GPU: Runtime → Change runtime type → GPU (T4 or better)

In [ ]:
# Cell 1: Check GPU
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# Define paths
DRIVE_BASE = '/content/drive/MyDrive/physics_llm'
DATA_PATH = f'{DRIVE_BASE}/data'
CHECKPOINT_PATH = f'{DRIVE_BASE}/checkpoints'

# Create directories
os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

# Check for required files
HDF5_DATA_PATH = f'{DATA_PATH}/clevrer_training_expanded.h5'
ZIP_PATH = f'{DRIVE_BASE}/physics_llm_colab.zip'
PHYSICS_CHECKPOINT = f'{CHECKPOINT_PATH}/physics_former_best.pt'

print(f"Checking required files:")
print(f"  HDF5 Data (152K samples): {'✅' if os.path.exists(HDF5_DATA_PATH) else '❌ MISSING'} {HDF5_DATA_PATH}")
print(f"  Code Zip: {'✅' if os.path.exists(ZIP_PATH) else '❌ MISSING'} {ZIP_PATH}")
print(f"  Physics Model: {'✅' if os.path.exists(PHYSICS_CHECKPOINT) else '❌ MISSING'} {PHYSICS_CHECKPOINT}")

# Verify all required files exist
missing = []
if not os.path.exists(HDF5_DATA_PATH):
    missing.append('clevrer_training_expanded.h5')
if not os.path.exists(ZIP_PATH):
    missing.append('physics_llm_colab.zip')
if not os.path.exists(PHYSICS_CHECKPOINT):
    missing.append('physics_former_best.pt')

if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print(f"\n✅ All files found! Checkpoints will save to: {CHECKPOINT_PATH}")

In [ ]:
# Cell 3: Extract code from zip
%cd /content

zip_path = '/content/drive/MyDrive/physics_llm/physics_llm_colab.zip'

!unzip -q -o {zip_path} -d /content/
print("✅ Code extracted!")

# List extracted contents
print("\nExtracted directories:")
!ls -la /content/ | grep -E "^d"

In [ ]:
# Cell 4: Install dependencies
!pip install torch transformers tqdm h5py --quiet
print("✅ Dependencies installed!")

In [ ]:
# Cell 5: Setup paths and imports
import os
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from pathlib import Path
import random
import h5py
import numpy as np

# Auto-detect where the zip extracted by walking /content to find a dir
# that contains both physics_former/ and physics_llm_adapter/. This works
# whether the zip's top-level is physics_llm_colab/, compsac_2026_code/,
# or extracts flat.
_IGNORED = {'drive', '.config', 'sample_data', '__pycache__', '.git'}
CODE_ROOT = None
for _root, _dirs, _ in os.walk('/content'):
    _dirs[:] = [d for d in _dirs if d not in _IGNORED]
    if 'physics_former' in _dirs and 'physics_llm_adapter' in _dirs:
        CODE_ROOT = _root
        break

if CODE_ROOT is None:
    raise RuntimeError(
        "Couldn't locate physics_former/ and physics_llm_adapter/ under /content. "
        "Did Cell 3 run successfully? Check the zip contents."
    )

print(f"✅ Code root: {CODE_ROOT}")

# Add extracted code to Python path. physics_former/ is on sys.path so
# imports like `from training.models.physics_former_full import ...` work;
# physics_llm_adapter/ is on sys.path so `from adapter_v2 import ...` works.
sys.path.insert(0, os.path.join(CODE_ROOT, 'physics_former'))
sys.path.insert(0, os.path.join(CODE_ROOT, 'physics_llm_adapter'))

# Paths
DRIVE_BASE = '/content/drive/MyDrive/physics_llm'
DATA_PATH = f'{DRIVE_BASE}/data'
CHECKPOINT_PATH = f'{DRIVE_BASE}/checkpoints'
HDF5_DATA_PATH = f'{DATA_PATH}/clevrer_training_expanded.h5'

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Verify extracted directories exist
_pf = os.path.join(CODE_ROOT, 'physics_former')
_pla = os.path.join(CODE_ROOT, 'physics_llm_adapter')
_training = os.path.join(_pf, 'training')
print(f"\n✅ Checking extracted code:")
print(f"   physics_former:      {'✅' if os.path.exists(_pf) else '❌'} {_pf}")
print(f"   physics_llm_adapter: {'✅' if os.path.exists(_pla) else '❌'} {_pla}")
print(f"   training module:     {'✅' if os.path.exists(_training) else '❌'} {_training}")

# Import from extracted code
from adapter_v2 import PhysicsLLMAdapterV2, create_adapter_v2
from train_adapter_v2 import collate_fn, evaluate, train_phase, PHYSICS_QUESTION_TYPES

print(f"\n✅ Imported modules successfully")


In [ ]:
# Cell 6: Load HDF5 Data (152K samples)

print(f'Loading CLEVRER HDF5 data from {HDF5_DATA_PATH}...')

# HDF5 Dataset class for efficient loading
class CLEVRERHDF5Dataset(Dataset):
    """Dataset for loading CLEVRER training data from HDF5 format."""

    def __init__(self, hdf5_path, max_samples=None):
        self.hdf5_path = hdf5_path
        self.hf = h5py.File(hdf5_path, 'r')

        self.num_samples = self.hf.attrs['num_samples']
        self.seq_len = self.hf.attrs['seq_len']
        self.num_objects = self.hf.attrs['num_objects']
        self.state_dim = self.hf.attrs['state_dim']

        self.indices = list(range(min(self.num_samples, max_samples or self.num_samples)))
        print(f'  Loaded {len(self.indices):,} samples')
        print(f'  State shape: ({self.seq_len}, {self.num_objects}, {self.state_dim})')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]

        states = torch.from_numpy(self.hf['states'][real_idx].astype(np.float32))
        mask = torch.from_numpy(self.hf['masks'][real_idx].astype(np.float32))

        question = self.hf['questions'][real_idx]
        if isinstance(question, bytes):
            question = question.decode('utf-8')

        answer = self.hf['answers'][real_idx]
        if isinstance(answer, bytes):
            answer = answer.decode('utf-8')

        question_type = self.hf['question_types'][real_idx]
        if isinstance(question_type, bytes):
            question_type = question_type.decode('utf-8')

        metadata_str = self.hf['metadata'][real_idx]
        if isinstance(metadata_str, bytes):
            metadata_str = metadata_str.decode('utf-8')
        metadata = json.loads(metadata_str)

        numerical_targets = self.hf['numerical_targets'][real_idx]

        return {
            'states': states,
            'mask': mask,
            'question': question,
            'answer': answer,
            'question_type': question_type,
            'metadata': metadata,
            'numerical_targets': {
                'count': float(numerical_targets[0]),
                'value': float(numerical_targets[1])
            }
        }

    def close(self):
        self.hf.close()

# Load dataset
full_dataset = CLEVRERHDF5Dataset(HDF5_DATA_PATH)

# Create train/test split (90/10)
total_size = len(full_dataset)
train_size = int(0.9 * total_size)

train_indices = list(range(train_size))
test_indices = list(range(train_size, total_size))

random.seed(42)
random.shuffle(train_indices)

class SubsetDataset(Dataset):
    def __init__(self, base_dataset, indices):
        self.base = base_dataset
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        return self.base[self.indices[idx]]

train_dataset = SubsetDataset(full_dataset, train_indices)
test_dataset = SubsetDataset(full_dataset, test_indices)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f'\n✅ Data loaded:')
print(f'   Train: {len(train_dataset):,} samples ({len(train_loader):,} batches)')
print(f'   Test: {len(test_dataset):,} samples ({len(test_loader):,} batches)')

# Show question type distribution
from collections import Counter
qtypes = Counter()
for i in range(min(1000, len(full_dataset))):
    qtypes[full_dataset[i]['question_type']] += 1
print(f'\nQuestion type distribution (sample of 1000):')
for qt, count in sorted(qtypes.items()):
    print(f'  {qt}: {count}')

In [ ]:
# Cell 7: Create Model

physics_checkpoint_path = '/content/drive/MyDrive/physics_llm/checkpoints/physics_former_best.pt'

print(f"Loading physics checkpoint: {physics_checkpoint_path}")

# Load physics model
from training.models.physics_former_full import FullPhysicsFormer
from training.configs.config import TrainingConfig

config = TrainingConfig()
checkpoint = torch.load(physics_checkpoint_path, map_location=device, weights_only=False)
model_state = checkpoint['model_state_dict']

has_orig_mod = any(k.startswith('_orig_mod.') for k in model_state.keys())
prefix = '_orig_mod.' if has_orig_mod else ''

schema_key = f'{prefix}schema_classifier.3.bias'
num_schema_classes = model_state[schema_key].shape[0]
hidden_dim_key = f'{prefix}transformer_layers.0.attention.q_proj.weight'
hidden_dim = model_state[hidden_dim_key].shape[0]
num_layers = sum(1 for k in model_state.keys() if f'{prefix}transformer_layers.' in k and '.attention.q_proj.weight' in k)
num_heads_key = f'{prefix}transformer_layers.0.attention.attention_bias_net.2.bias'
num_heads = model_state[num_heads_key].shape[0]

print(f"   Detected: hidden_dim={hidden_dim}, num_layers={num_layers}, num_heads={num_heads}, schema_classes={num_schema_classes}")

physics_model = FullPhysicsFormer(
    state_dim=config.state_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    num_heads=num_heads,
    ff_dim=hidden_dim * 4,
    max_objects=config.max_objects,
    dropout=config.dropout,
    num_schema_classes=num_schema_classes
).to(device)

if has_orig_mod:
    cleaned_state = {k.replace('_orig_mod.', ''): v for k, v in model_state.items()}
else:
    cleaned_state = model_state
filtered_state = {k: v for k, v in cleaned_state.items()
                  if 'rope.cos_cached' not in k and 'rope.sin_cached' not in k}
physics_model.load_state_dict(filtered_state, strict=False)
print("✅ Loaded PhysicsFormer checkpoint")

adapter = create_adapter_v2(
    physics_model=physics_model,
    physics_dim=hidden_dim,
    num_prefix_tokens=64,
    freeze_physics=True,
    freeze_llm=True
).to(device)

print(f'\n✅ Adapter created on {device}')
print(f'   Total parameters: {sum(p.numel() for p in adapter.parameters()):,}')
print(f'   Trainable parameters: {sum(p.numel() for p in adapter.parameters() if p.requires_grad):,}')

In [ ]:
# Cell 8: List Available Checkpoints (for resuming)

os.makedirs(CHECKPOINT_PATH, exist_ok=True)

print("Available checkpoints:")
checkpoints = sorted([f for f in os.listdir(CHECKPOINT_PATH) if f.endswith('.pt')])
if checkpoints:
    for cp in checkpoints:
        size_mb = os.path.getsize(os.path.join(CHECKPOINT_PATH, cp)) / 1e6
        print(f"  - {cp} ({size_mb:.1f} MB)")
else:
    print("  No checkpoints found. Starting fresh.")

In [ ]:
# Cell 9: Resume Configuration + Auto-resume helper
#
# Two kinds of resume are supported:
#   (A) Phase-level resume  -- user-controlled. Set RESUME_CHECKPOINT + START_PHASE
#       below to load a finished-phase checkpoint and jump into a later phase.
#   (B) Mid-phase crash recovery -- automatic. Each phase cell calls
#       resume_phase_state() AFTER building its optimizer, which scans
#       CHECKPOINT_PATH for the latest adapter_phase{N}_epoch*.pt, restores
#       model + optimizer + best_loss + early-stopping counter, and returns
#       (start_epoch, best_loss, patience_counter). train_phase() picks up
#       from start_epoch with the saved state.
#
# The combination means: if Colab kernels die mid-epoch, just re-run the
# notebook top-to-bottom with the same RESUME_CHECKPOINT / START_PHASE
# values -- the latest per-epoch checkpoint will be picked up automatically.

import re

# --- Phase-level resume (manual) ---
# Set this to jump to a later phase using a finished-phase checkpoint.
# Leave None to let mid-phase auto-resume handle things.
RESUME_CHECKPOINT = None  # e.g. f'{CHECKPOINT_PATH}/adapter_v2_expanded_final.pt'
START_PHASE = 1           # 1, 2, or 3 (phases numbered below 1..3)


def _latest_phase_epoch_checkpoint(checkpoint_dir, phase_num):
    """Return (path, epoch) for the highest-epoch adapter_phase{N}_epoch*.pt
    in ``checkpoint_dir``, or (None, 0) if none exist.

    Also considers adapter_phase{N}_complete_loss*.pt as a terminal marker:
    if present, returns it with epoch = -1 (sentinel meaning "phase done").
    """
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        return None, 0

    # Per-epoch candidates: adapter_phase{N}_epoch{E}_loss{L}.pt
    pattern = re.compile(rf"^adapter_phase{phase_num}_epoch(\d+)_loss[\d.]+\.pt$")
    best_path, best_epoch = None, -1
    for p in checkpoint_dir.glob(f"adapter_phase{phase_num}_epoch*_loss*.pt"):
        m = pattern.match(p.name)
        if m:
            e = int(m.group(1))
            if e > best_epoch:
                best_path, best_epoch = p, e

    # Phase-complete sentinel: adapter_phase{N}_complete_loss{L}.pt
    complete_candidates = sorted(checkpoint_dir.glob(f"adapter_phase{phase_num}_complete_loss*.pt"))
    if complete_candidates:
        # Use the newest complete file if it's newer than the best per-epoch file
        newest_complete = max(complete_candidates, key=lambda p: p.stat().st_mtime)
        if best_path is None or newest_complete.stat().st_mtime > best_path.stat().st_mtime:
            return newest_complete, -1  # -1 = phase already finished

    return best_path, max(best_epoch, 0)


def resume_phase_state(model, optimizer, checkpoint_dir, phase_num, device):
    """Auto-resume Phase ``phase_num`` from the latest per-epoch checkpoint.

    Call this AFTER ``model.set_training_phase(...)`` and AFTER the optimizer
    is constructed (so LoRA parameters and the correct trainable-param set
    already exist).

    Returns a dict with:
        start_epoch, best_loss, epochs_without_improvement, resumed_from
    ``resumed_from`` is None if no prior checkpoint was found (fresh start).
    ``start_epoch = 0, best_loss = inf, epochs_without_improvement = 0`` on
    fresh start. If a phase-complete sentinel is present, returns
    ``start_epoch = -1`` so the caller can skip the phase entirely.
    """
    path, epoch = _latest_phase_epoch_checkpoint(checkpoint_dir, phase_num)
    if path is None:
        print(f"[resume] Phase {phase_num}: no prior checkpoint -- fresh start")
        return {'start_epoch': 0, 'best_loss': None,
                'epochs_without_improvement': 0, 'resumed_from': None}

    if epoch == -1:
        print(f"[resume] Phase {phase_num}: found phase-complete sentinel {path.name} -- skipping phase")
        # Load weights so downstream phases see the finished state
        ckpt = torch.load(path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'], strict=False)
        return {'start_epoch': -1, 'best_loss': ckpt.get('best_loss'),
                'epochs_without_improvement': 0, 'resumed_from': str(path)}

    print(f"[resume] Phase {phase_num}: loading {path.name} (epoch {epoch})")
    ckpt = torch.load(path, map_location=device, weights_only=False)
    # strict=False tolerates stale LoRA keys from a previous rank or optional
    # heads added since the checkpoint was written.
    missing, unexpected = model.load_state_dict(ckpt['model_state_dict'], strict=False)
    if missing:
        print(f"  [resume] missing keys: {len(missing)} (first 3: {missing[:3]})")
    if unexpected:
        print(f"  [resume] unexpected keys: {len(unexpected)} (first 3: {unexpected[:3]})")
    if 'optimizer_state_dict' in ckpt:
        try:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            print(f"  [resume] optimizer state restored")
        except (ValueError, KeyError) as e:
            print(f"  [resume] could NOT restore optimizer state ({e}); "
                  f"optimizer will use its default warm-up state")
    return {
        'start_epoch': int(ckpt.get('epoch', epoch)),
        'best_loss': ckpt.get('best_loss'),
        'epochs_without_improvement': int(ckpt.get('epochs_without_improvement', 0)),
        'resumed_from': str(path),
    }


# --- Apply phase-level RESUME_CHECKPOINT if set ---
if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    print(f'Loading phase-level checkpoint: {RESUME_CHECKPOINT}')
    ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
    # If the checkpoint was trained with LoRA, we need to apply LoRA BEFORE loading
    # so the target module topology matches the state_dict keys.
    has_lora_keys = any('lora_' in k for k in ckpt['model_state_dict'].keys())
    if has_lora_keys and not adapter.has_lora:
        print('  Checkpoint contains LoRA tensors -- applying LoRA to adapter before load')
        adapter.apply_lora(rank=8, alpha=16.0)
    missing, unexpected = adapter.load_state_dict(ckpt['model_state_dict'], strict=False)
    if missing:
        print(f'  missing keys: {len(missing)} (first 3: {missing[:3]})')
    if unexpected:
        print(f'  unexpected keys: {len(unexpected)} (first 3: {unexpected[:3]})')
    loaded_phase = ckpt.get('phase', 1)
    loaded_epoch = ckpt.get('epoch', 0)
    loaded_loss = ckpt.get('loss', ckpt.get('best_loss', 0.0))
    print(f'✅ Loaded from Phase {loaded_phase}, Epoch {loaded_epoch}, Loss {loaded_loss:.4f}')
    print(f'   Starting from Phase {START_PHASE}')
else:
    print(f'Starting fresh from Phase {START_PHASE} '
          f'(mid-phase crash recovery will still scan CHECKPOINT_PATH automatically)')


In [ ]:
# Cell 10: Phase 1 - Adapter + Numerical + Descriptive Heads
#
# Auto-resumes from the latest adapter_phase1_epoch*.pt if a prior run
# crashed mid-phase. If adapter_phase1_complete_loss*.pt exists the phase
# is skipped and the finished weights are loaded.

checkpoint_dir = Path(CHECKPOINT_PATH)

if START_PHASE <= 1:
    print("[PHASE] Training adapter + numerical + descriptive heads")
    adapter.set_training_phase('adapter')
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, adapter.parameters()),
        lr=1e-4, weight_decay=0.01
    )

    # Auto-resume: scan checkpoint_dir for the latest Phase-1 checkpoint
    resume = resume_phase_state(adapter, optimizer, checkpoint_dir, phase_num=1, device=device)

    if resume['start_epoch'] == -1:
        print('[PHASE 1] Skipping -- phase-complete sentinel found, weights loaded.')
    else:
        train_phase(
            adapter, train_loader, optimizer, device,
            phase_name="Phase 1: Adapter + Numerical + Descriptive",
            max_epochs=20, patience=5, min_delta=0.001,
            checkpoint_dir=checkpoint_dir, phase_num=1, save_every_n_epochs=3,
            start_epoch=resume['start_epoch'],
            initial_best_loss=resume['best_loss'],
            initial_epochs_without_improvement=resume['epochs_without_improvement'],
        )
else:
    print(f'[SKIP] Phase 1 (starting from Phase {START_PHASE})')


In [ ]:
# Cell 11: Phase 2 - LoRA on LLM attention + contrastive loss
#
# Matches the Jan 25 baseline (adapter_v2_distilgpt2_contrastive.pt) recipe:
#   - LoRA (rank=8) adapts the LLM attention/MLP projections so the frozen
#     distilgpt2 learns to condition on the 64 physics prefix tokens.
#     Without this, perplexity-ranking MCQ choices collapses to text priors
#     (~50% on 2-choice questions).
#   - Contrastive loss forces prefix tokens built from real vs zeroed physics
#     states to be different, preventing the adapter from collapsing to an
#     ignore-the-physics degenerate solution.
#
# Resume behavior:
#   - To jump from a finished Phase 1 checkpoint, set in Cell 9:
#       RESUME_CHECKPOINT = f'{CHECKPOINT_PATH}/adapter_v2_expanded_final.pt'
#       START_PHASE = 2
#   - Mid-phase crashes are recovered automatically from the latest
#     adapter_phase2_epoch*.pt in CHECKPOINT_PATH (LoRA tensors already
#     exist after set_training_phase('lora'), so state_dict load works).

if START_PHASE <= 2:
    adapter.set_training_phase('lora', lora_rank=8, lora_alpha=16.0)

    # LoRA adds new trainable params (lora_A / lora_B) -- rebuild the optimizer
    # so they are picked up. Only unfrozen params (adapter MLP + numerical +
    # descriptive heads + LoRA tensors) flow through the filter.
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, adapter.parameters()),
        lr=5e-5, weight_decay=0.01
    )

    # Auto-resume: scan for the latest Phase-2 mid-epoch checkpoint.
    resume = resume_phase_state(adapter, optimizer, checkpoint_dir, phase_num=2, device=device)

    if resume['start_epoch'] == -1:
        print('[PHASE 2] Skipping -- phase-complete sentinel found, weights loaded.')
    else:
        train_phase(
            adapter, train_loader, optimizer, device,
            phase_name="Phase 2: LoRA + Contrastive",
            max_epochs=15, patience=4, min_delta=0.0005,
            checkpoint_dir=checkpoint_dir, phase_num=2, save_every_n_epochs=3,
            use_contrastive=True, contrastive_weight=0.1,
            start_epoch=resume['start_epoch'],
            initial_best_loss=resume['best_loss'],
            initial_epochs_without_improvement=resume['epochs_without_improvement'],
        )
else:
    print(f'[SKIP] Phase 2 (starting from Phase {START_PHASE})')


In [ ]:
# Cell 12: Phase 3 - Full LLM fine-tuning (optional)
#
# Now gated on START_PHASE <= 3 for consistency with Cells 10 and 11.
# Auto-resumes from adapter_phase3_epoch*.pt on crash; skips if the
# phase-complete sentinel exists.

if START_PHASE <= 3:
    adapter.set_training_phase('llm_full')
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, adapter.parameters()),
        lr=2e-5, weight_decay=0.01
    )

    resume = resume_phase_state(adapter, optimizer, checkpoint_dir, phase_num=3, device=device)

    if resume['start_epoch'] == -1:
        print('[PHASE 3] Skipping -- phase-complete sentinel found, weights loaded.')
    else:
        train_phase(
            adapter, train_loader, optimizer, device,
            phase_name="Phase 3: Full LLM",
            max_epochs=10, patience=3, min_delta=0.0005,
            checkpoint_dir=checkpoint_dir, phase_num=3, save_every_n_epochs=3,
            start_epoch=resume['start_epoch'],
            initial_best_loss=resume['best_loss'],
            initial_epochs_without_improvement=resume['epochs_without_improvement'],
        )
else:
    print(f'[SKIP] Phase 3 (starting from Phase {START_PHASE})')


In [ ]:
# Cell 13: Save Final Model

final_path = f'{CHECKPOINT_PATH}/adapter_v2_expanded_final.pt'
torch.save({
    'model_state_dict': adapter.state_dict(),
    'llm_name': 'distilgpt2',
    'physics_dim': hidden_dim,
    'num_prefix_tokens': 64,
    'training_samples': len(train_dataset),
}, final_path)
print(f'\n✅ Final model saved to: {final_path}')

In [ ]:
# Cell 14: Evaluation

accuracy = evaluate(adapter, test_loader, device, num_samples=500)
print(f'\n✅ Final Accuracy: {accuracy:.1f}%')

In [ ]:
# Cell 15: List All Saved Checkpoints

print('Saved checkpoints:')
for f in sorted(Path(CHECKPOINT_PATH).glob('*.pt')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name} ({size_mb:.1f} MB)')

In [ ]:
# Cell 17: List All Saved Checkpoints

print('Saved checkpoints:')
for f in sorted(Path(CHECKPOINT_PATH).glob('*.pt')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name} ({size_mb:.1f} MB)')

In [ ]:
# Cell 19: Zero-Shot Metaphor Evaluation
# Tests whether physics grounding enables metaphor understanding WITHOUT training on metaphors

METAPHOR_QUESTION_TYPES = [
    # Physics metaphors
    "metaphor_collision", "metaphor_momentum", "metaphor_equilibrium",
    "metaphor_trajectory", "metaphor_force",
    # Lakoff image schemas
    "metaphor_container", "metaphor_source_path_goal", "metaphor_balance",
    "metaphor_link", "metaphor_center_periphery", "metaphor_resistance",
    # Mathematical metaphors (Lakoff & Núñez)
    "metaphor_arithmetic_motion", "metaphor_arithmetic_collection",
    "metaphor_measuring_stick", "metaphor_change_motion", "metaphor_numbers_points",
]

METAPHOR_TEMPLATES = {
    "metaphor_collision": [
        ("If these objects were people in a debate, what would happen when they meet?",
         lambda s, m: _metaphor_collision_answer(s, m)),
    ],
    "metaphor_momentum": [
        ("If this object's momentum represented career progress, describe it.",
         lambda s, m: _metaphor_momentum_answer(s, m)),
    ],
    "metaphor_equilibrium": [
        ("If this system were a relationship, is it balanced or unstable?",
         lambda s, m: _metaphor_equilibrium_answer(s, m)),
    ],
    "metaphor_trajectory": [
        ("If this trajectory were a person's life path, where are they heading?",
         lambda s, m: _metaphor_trajectory_answer(s, m)),
    ],
    "metaphor_force": [
        ("If the forces on this object were social pressures, describe the situation.",
         lambda s, m: _metaphor_force_answer(s, m)),
    ],
    "metaphor_container": [
        ("If this region were a container for ideas, describe what's inside vs outside.",
         lambda s, m: _metaphor_container_answer(s, m)),
    ],
    "metaphor_balance": [
        ("If this system's balance represented justice, is it fair?",
         lambda s, m: _metaphor_balance_answer(s, m)),
    ],
}

def _get_physics_state(states, masks):
    """Extract key physics properties from states."""
    last_state = states[-1] if states.dim() == 3 else states
    active_mask = masks > 0.5 if masks.dim() == 1 else masks[0] > 0.5

    positions = last_state[:, 0:3]
    velocities = last_state[:, 3:6] if last_state.shape[-1] > 5 else torch.zeros_like(positions)

    speeds = torch.norm(velocities, dim=-1)
    max_speed = speeds[active_mask].max().item() if active_mask.any() else 0

    # Check for approaching objects
    n_objects = active_mask.sum().item()
    approaching = False
    if n_objects >= 2:
        active_idx = torch.where(active_mask)[0]
        for i in range(len(active_idx)):
            for j in range(i+1, len(active_idx)):
                rel_pos = positions[active_idx[j]] - positions[active_idx[i]]
                rel_vel = velocities[active_idx[j]] - velocities[active_idx[i]]
                closing = -torch.dot(rel_vel, rel_pos / (torch.norm(rel_pos) + 1e-6))
                if closing > 0.05:
                    approaching = True

    return {
        'max_speed': max_speed,
        'n_objects': n_objects,
        'approaching': approaching,
        'total_momentum': (speeds * active_mask.float()).sum().item(),
    }

def _metaphor_collision_answer(states, masks):
    props = _get_physics_state(states, masks)
    if props['approaching']:
        if props['max_speed'] > 0.5:
            return "An intense clash of ideas - both sides coming in with strong convictions"
        else:
            return "A measured exchange - both parties approaching cautiously"
    return "Parallel paths - the ideas don't directly engage"

def _metaphor_momentum_answer(states, masks):
    props = _get_physics_state(states, masks)
    if props['max_speed'] > 0.5:
        return "Strong career momentum - rapid advancement and clear direction"
    elif props['max_speed'] > 0.1:
        return "Steady progress - consistent forward movement"
    return "Career at a standstill - waiting for the next opportunity"

def _metaphor_equilibrium_answer(states, masks):
    props = _get_physics_state(states, masks)
    if props['max_speed'] < 0.05:
        return "A stable, balanced relationship - both parties at rest"
    elif props['approaching']:
        return "Tension building - the relationship is heading toward a confrontation"
    return "Dynamic but stable - movement without conflict"

def _metaphor_trajectory_answer(states, masks):
    props = _get_physics_state(states, masks)
    if props['max_speed'] > 0.3:
        return "A life in motion - heading somewhere with purpose and drive"
    elif props['max_speed'] > 0.05:
        return "Gradual progress - taking life one step at a time"
    return "At a crossroads - paused to consider the next direction"

def _metaphor_force_answer(states, masks):
    props = _get_physics_state(states, masks)
    if props['n_objects'] > 3:
        return "Multiple competing pressures - pulled in different directions by various influences"
    elif props['approaching']:
        return "Pressure building from one direction - a dominant influence approaching"
    return "Relatively free from external pressure - space to make independent choices"

def _metaphor_container_answer(states, masks):
    props = _get_physics_state(states, masks)
    return f"The container holds {props['n_objects']} distinct concepts, each with its own momentum and trajectory"

def _metaphor_balance_answer(states, masks):
    props = _get_physics_state(states, masks)
    if props['max_speed'] < 0.05:
        return "Justice is balanced - all parties at equilibrium"
    return "The scales are tipping - some have more momentum than others"


def generate_metaphor_qa(states, masks, num_samples=100):
    """Generate metaphor QA pairs for zero-shot evaluation."""
    qa_pairs = []
    import random

    for _ in range(num_samples):
        q_type = random.choice(list(METAPHOR_TEMPLATES.keys()))
        template, answer_fn = random.choice(METAPHOR_TEMPLATES[q_type])

        answer = answer_fn(states, masks)
        qa_pairs.append({
            'states': states,
            'masks': masks,
            'question': template,
            'answer': answer,
            'question_type': q_type
        })

    return qa_pairs


def evaluate_zero_shot_metaphor(model, test_pairs, device, num_samples=100):
    """Evaluate model on metaphor questions it was NOT trained on."""
    model.eval()

    results = {qt: {'correct': 0, 'total': 0, 'semantic_sim': []} for qt in METAPHOR_TEMPLATES.keys()}

    print(f"\n{'='*60}")
    print("ZERO-SHOT METAPHOR EVALUATION")
    print("Testing if physics grounding enables metaphor understanding")
    print(f"{'='*60}")

    with torch.no_grad():
        for i, pair in enumerate(test_pairs[:num_samples]):
            states = pair['states'].unsqueeze(0).to(device) if pair['states'].dim() == 3 else pair['states'].unsqueeze(0).unsqueeze(0).to(device)
            masks = pair['masks'].unsqueeze(0).to(device) if pair['masks'].dim() == 1 else pair['masks'].to(device)
            question = pair['question']
            expected = pair['answer']
            q_type = pair['question_type']

            # Handle 3D states -> 4D for model
            if states.dim() == 3:
                states = states.unsqueeze(1)  # Add time dimension

            predicted = model(states, masks, [question])[0]

            # Compute semantic similarity (keyword overlap)
            pred_words = set(predicted.lower().split())
            exp_words = set(expected.lower().split())
            overlap = len(pred_words & exp_words) / max(len(exp_words), 1)

            results[q_type]['total'] += 1
            results[q_type]['semantic_sim'].append(overlap)
            if overlap > 0.2:  # Threshold for "correct"
                results[q_type]['correct'] += 1

    # Print results
    print(f"\n{'Question Type':<30} {'Accuracy':<15} {'Avg Similarity':<15}")
    print("-" * 60)

    total_correct = 0
    total_count = 0

    for q_type, r in results.items():
        if r['total'] > 0:
            acc = 100 * r['correct'] / r['total']
            avg_sim = sum(r['semantic_sim']) / len(r['semantic_sim'])
            print(f"{q_type:<30} {acc:>12.1f}% {avg_sim:>14.3f}")
            total_correct += r['correct']
            total_count += r['total']

    overall_acc = 100 * total_correct / max(total_count, 1)
    print("-" * 60)
    print(f"{'OVERALL':<30} {overall_acc:>12.1f}%")
    print(f"{'='*60}")

    return overall_acc, results


# Run zero-shot evaluation
print("Generating metaphor test pairs from training data...")
sample_batch = next(iter(test_loader))
metaphor_test_pairs = generate_metaphor_qa(
    sample_batch['states'][0],
    sample_batch['masks'][0],
    num_samples=50
)

# Expand to use multiple samples from test set
all_metaphor_pairs = []
for batch in test_loader:
    if len(all_metaphor_pairs) >= 200:
        break
    for i in range(min(batch['states'].shape[0], 5)):
        pairs = generate_metaphor_qa(batch['states'][i], batch['masks'][i], num_samples=10)
        all_metaphor_pairs.extend(pairs)

print(f"Generated {len(all_metaphor_pairs)} metaphor test pairs")
zero_shot_acc, zero_shot_results = evaluate_zero_shot_metaphor(
    adapter, all_metaphor_pairs, device, num_samples=200
)

In [ ]:
# Cell 20: CLEVRER-Style Reasoning Evaluation
# Tests causal, predictive, and counterfactual reasoning capabilities

CLEVRER_QUESTION_TYPES = {
    'explanatory': [
        ("What caused the current motion pattern?", 'causal_chain'),
        ("Explain the causal sequence leading to this state.", 'causal_chain'),
        ("What events led to this configuration?", 'causal_chain'),
    ],
    'predictive': [
        ("What will happen next in this scene?", 'future_prediction'),
        ("Predict the next significant event.", 'future_prediction'),
        ("Will any objects collide?", 'collision_prediction'),
    ],
    'counterfactual': [
        ("What would happen if object 1 were removed?", 'counterfactual'),
        ("How would the outcome change without the fastest object?", 'counterfactual'),
        ("If object 1 had not moved, what would be different?", 'counterfactual'),
    ],
}

def generate_clevrer_answer(states, masks, q_type):
    """Generate ground truth answer based on physics state."""
    props = _get_physics_state(states, masks)

    if q_type == 'causal_chain':
        if props['max_speed'] > 0.1:
            return "Objects received impulses causing motion; momentum transfer from collisions"
        return "Objects appear to be in initial or steady state - no significant causal events"

    elif q_type == 'future_prediction':
        if props['approaching']:
            return "Collision predicted between approaching objects"
        elif props['max_speed'] > 0.05:
            return f"{props['n_objects']} object(s) will continue along current trajectories"
        return "System appears stable - objects will remain stationary"

    elif q_type == 'collision_prediction':
        if props['approaching']:
            return "Yes, objects are on collision course"
        return "No collision imminent"

    elif q_type == 'counterfactual':
        if props['n_objects'] > 1 and props['max_speed'] > 0.1:
            return "Removing the object would prevent momentum transfer; other objects unaffected"
        return "Removing object 1 would have minimal effect on the system"

    return "Unable to determine"


def generate_clevrer_qa(states, masks, num_samples=50):
    """Generate CLEVRER-style QA pairs."""
    qa_pairs = []
    import random

    for _ in range(num_samples):
        category = random.choice(list(CLEVRER_QUESTION_TYPES.keys()))
        question, q_type = random.choice(CLEVRER_QUESTION_TYPES[category])
        answer = generate_clevrer_answer(states, masks, q_type)

        qa_pairs.append({
            'states': states,
            'masks': masks,
            'question': question,
            'answer': answer,
            'question_type': q_type,
            'clevrer_category': category
        })

    return qa_pairs


def evaluate_clevrer_reasoning(model, test_pairs, device, num_samples=100):
    """Evaluate CLEVRER-style reasoning capabilities."""
    model.eval()

    results = {cat: {'correct': 0, 'total': 0} for cat in CLEVRER_QUESTION_TYPES.keys()}

    print(f"\n{'='*60}")
    print("CLEVRER-STYLE REASONING EVALUATION")
    print("Testing explanatory, predictive, and counterfactual reasoning")
    print(f"{'='*60}")

    with torch.no_grad():
        for pair in test_pairs[:num_samples]:
            states = pair['states'].unsqueeze(0).to(device) if pair['states'].dim() == 3 else pair['states'].unsqueeze(0).unsqueeze(0).to(device)
            masks = pair['masks'].unsqueeze(0).to(device) if pair['masks'].dim() == 1 else pair['masks'].to(device)

            if states.dim() == 3:
                states = states.unsqueeze(1)

            question = pair['question']
            expected = pair['answer']
            category = pair['clevrer_category']

            predicted = model(states, masks, [question])[0]

            # Check for key concept overlap
            pred_lower = predicted.lower()
            exp_lower = expected.lower()

            # Flexible matching for yes/no questions
            is_correct = False
            if 'yes' in exp_lower and ('yes' in pred_lower or 'will' in pred_lower or 'collision' in pred_lower):
                is_correct = True
            elif 'no' in exp_lower and ('no' in pred_lower or "won't" in pred_lower or 'stable' in pred_lower):
                is_correct = True
            elif any(word in pred_lower for word in exp_lower.split()[:5]):
                is_correct = True

            results[category]['total'] += 1
            if is_correct:
                results[category]['correct'] += 1

    print(f"\n{'CLEVRER Category':<20} {'Accuracy':<15} {'Correct/Total':<15}")
    print("-" * 50)

    total_correct = 0
    total_count = 0

    for cat, r in results.items():
        if r['total'] > 0:
            acc = 100 * r['correct'] / r['total']
            print(f"{cat:<20} {acc:>12.1f}% {r['correct']:>6}/{r['total']:<6}")
            total_correct += r['correct']
            total_count += r['total']

    overall_acc = 100 * total_correct / max(total_count, 1)
    print("-" * 50)
    print(f"{'OVERALL':<20} {overall_acc:>12.1f}%")
    print(f"{'='*60}")

    return overall_acc, results


# Generate CLEVRER test pairs
all_clevrer_pairs = []
for batch in test_loader:
    if len(all_clevrer_pairs) >= 150:
        break
    for i in range(min(batch['states'].shape[0], 5)):
        pairs = generate_clevrer_qa(batch['states'][i], batch['masks'][i], num_samples=10)
        all_clevrer_pairs.extend(pairs)

print(f"Generated {len(all_clevrer_pairs)} CLEVRER-style test pairs")
clevrer_acc, clevrer_results = evaluate_clevrer_reasoning(
    adapter, all_clevrer_pairs, device, num_samples=150
)

In [ ]:
# Cell 21: Summary of All Evaluations

print("\n" + "=" * 70)
print("COMPLETE EVALUATION SUMMARY")
print("=" * 70)

print(f"\n📊 STANDARD QA ACCURACY (trained question types):")
print(f"   Overall: {accuracy:.1f}%")

print(f"\n🎯 ZERO-SHOT METAPHOR EVALUATION (untrained):")
print(f"   Overall: {zero_shot_acc:.1f}%")
print("   This tests Lakoff's embodied cognition hypothesis:")
print("   Does physics grounding enable metaphor understanding?")

print(f"\n🔬 CLEVRER-STYLE REASONING:")
print(f"   Overall: {clevrer_acc:.1f}%")
for cat, r in clevrer_results.items():
    if r['total'] > 0:
        acc = 100 * r['correct'] / r['total']
        print(f"   - {cat}: {acc:.1f}%")

print("\n" + "=" * 70)
print("KEY INSIGHTS:")
print("=" * 70)

if zero_shot_acc > 30:
    print("✅ Physics grounding shows transfer to metaphor understanding")
else:
    print("⚠️ Limited metaphor transfer - may need more physics diversity")

if clevrer_acc > 50:
    print("✅ Strong causal/predictive reasoning capabilities")
else:
    print("⚠️ Causal reasoning needs improvement")

print("\n📁 All checkpoints saved to Google Drive:")
print(f"   {CHECKPOINT_PATH}")
print("=" * 70)